In [ ]:
import pandas as pd
import numpy as np
import re

# Superseded by process_autumn.py (kept for reference)
RAW_PATH = '../data/raw/autumn/autumn.csv'

df_raw = pd.read_csv(RAW_PATH, sep=';', encoding='utf-8-sig', dtype=str)
print(f"Raw shape: {df_raw.shape}")
print("First 8 columns:", df_raw.columns[:8].tolist())

In [ ]:
columns_info = []
for col in df_raw.columns:
    m = re.match(r'^(\d+)\.\s*(.*)', col)
    if m:
        qnum = int(m.group(1))
        rest = m.group(2)
        if ' - ' in rest:
            question_text, option_text = rest.split(' - ', 1)
        else:
            question_text = rest
            option_text = None
        columns_info.append({
            'col': col,
            'qnum': qnum,
            'question_text': question_text.strip(),
            'option_text': option_text.strip() if option_text else None,
        })
    else:
        columns_info.append({
            'col': col,
            'qnum': None,
            'question_text': col,
            'option_text': None,
        })

n_q = sum(1 for c in columns_info if c['qnum'] is not None)
print(f"Parsed {n_q} question columns")

In [ ]:
def get_block(qnum):
    """Return survey block (1-5) based on question number."""
    if 1  <= qnum <= 23:  return 1
    if 24 <= qnum <= 46:  return 2
    if 47 <= qnum <= 69:  return 3
    if 70 <= qnum <= 92:  return 4
    if 93 <= qnum <= 115: return 5
    return None

# Block 1 (freshman survey, full question set)
block1_map = {
    1: 'course', 2: 'program', 3: 'decision', 4: 'olympiads',
    5: 'english_level', 6: 'difficulties', 7: 'hours_planned',
    8: 'work_plans', 9: 'work_related', 10: 'travel_time',
    11: 'expectations', 12: 'priorities', 13: 'confidence',
    14: 'special_conditions', 15: 'conditions_desc', 16: 'contact',
    17: 'about_you', 18: 'games_team', 19: 'games_solo',
    20: 'games_mobile', 21: 'games_intellectual',
    22: 'tournament_interest', 23: 'extracurricular',
}

# Blocks 2-5 (returning students; same questions shifted by 23*(block-1))
block2_map = {
    1: 'program', 2: 'difficulties', 3: 'hours_planned',
    4: 'hours_last_year', 5: 'work_plans', 6: 'work_related',
    7: 'travel_time', 8: 'expectations', 9: 'priorities',
    10: 'confidence', 11: 'thoughts_transfer', 12: 'transfer_influences',
    13: 'missing', 14: 'special_conditions', 15: 'conditions_desc',
    16: 'contact', 17: 'about_you', 18: 'games_team', 19: 'games_solo',
    20: 'games_mobile', 21: 'games_intellectual',
    22: 'tournament_interest', 23: 'extracurricular',
}

# Deterministic order (list, not set)
multi_select_vars = [
    'difficulties', 'priorities', 'about_you',
    'games_team', 'games_solo', 'games_mobile', 'games_intellectual',
    'thoughts_transfer', 'missing',
]

# Final column order
variable_order = [
    'course', 'program', 'decision', 'olympiads', 'english_level',
    'difficulties', 'hours_planned', 'hours_last_year',
    'work_plans', 'work_related', 'travel_time',
    'expectations', 'priorities', 'confidence',
    'special_conditions', 'conditions_desc', 'contact',
    'about_you', 'games_team', 'games_solo', 'games_mobile',
    'games_intellectual', 'tournament_interest', 'extracurricular',
    'thoughts_transfer', 'transfer_influences', 'missing',
]

metadata_order = [
    'response_id', 'date', 'time_spent', 'ip',
    'source', 'os', 'browser', 'device',
]

assert set(multi_select_vars) <= set(variable_order), "multi_select_vars must be part of variable_order"

In [ ]:
unified_rows = []

for idx, row in df_raw.iterrows():
    # Seed every variable as NaN (single-choice) or [] (multi-select)
    unified = {var: np.nan for var in variable_order}
    for var in multi_select_vars:
        unified[var] = []

    for ci in columns_info:
        qnum = ci['qnum']
        if qnum is None:
            continue
        block = get_block(qnum)
        if block is None:
            continue

        if block == 1:
            rel = qnum
            var_map = block1_map
        else:
            rel = qnum - 23 * (block - 1)
            var_map = block2_map

        if rel not in var_map:
            continue

        name = var_map[rel]
        value = row[ci['col']]
        if pd.isna(value) or str(value).strip() == '':
            continue

        if name in multi_select_vars:
            if ci['option_text']:
                if ci['option_text'] == 'Впишите свой ответ':
                    unified[name].append(str(value).strip())
                else:
                    unified[name].append(ci['option_text'])
            else:
                unified[name].append(str(value).strip())
        else:
            # Keep the first non-empty value; subsequent duplicates ignored
            if pd.isna(unified[name]):
                unified[name] = str(value).strip()

    # Collapse multi-select lists to '; '-joined strings (or '' when empty)
    for var in multi_select_vars:
        items = unified[var]
        if items:
            seen, deduped = set(), []
            for x in items:
                if x not in seen:
                    seen.add(x); deduped.append(x)
            unified[var] = '; '.join(deduped)
        else:
            unified[var] = ''

    # Metadata
    unified['response_id'] = row['Номер ответа']
    unified['date']        = row['Дата ответа']
    unified['time_spent']  = row['Время выполнения']
    unified['ip']          = row['IP - адрес']
    unified['source']      = row['Источник распространения']
    unified['os']          = row['Операционная система']
    unified['browser']     = row['Браузер']
    unified['device']      = row['Устройство']

    unified_rows.append(unified)

df_clean = pd.DataFrame(unified_rows)
print(f"Unified shape: {df_clean.shape}")

In [ ]:
# Guarantee every column exists (all seeded in Cell 5, but be defensive)
for c in metadata_order + variable_order:
    if c not in df_clean.columns:
        df_clean[c] = np.nan
df_clean = df_clean[metadata_order + variable_order]

# Type conversions
df_clean['response_id'] = pd.to_numeric(df_clean['response_id'], errors='coerce').astype('Int64')
df_clean['time_spent']  = pd.to_numeric(df_clean['time_spent'], errors='coerce')
df_clean['date']        = pd.to_datetime(df_clean['date'], errors='coerce', dayfirst=True)
df_clean['confidence']  = pd.to_numeric(df_clean['confidence'], errors='coerce')

print("Final shape:", df_clean.shape)
print("Columns:", df_clean.columns.tolist())
print("\nNaN counts per column:")
print(df_clean.isna().sum().to_string())

In [ ]:
def strip_ws(s):
    """Trim + collapse whitespace; preserve NaN as NaN."""
    if pd.isna(s):
        return s
    return re.sub(r'\s+', ' ', str(s)).strip()

# Text columns (skip numeric/datetime)
text_cols = [c for c in df_clean.columns
             if c not in ('response_id', 'date', 'time_spent', 'confidence')]
for c in text_cols:
    df_clean[c] = df_clean[c].map(strip_ws)

# Canonicalise hour buckets: drop block-2 dot, unify dash
for c in ('hours_planned', 'hours_last_year'):
    df_clean[c] = (df_clean[c]
                   .astype('string')
                   .str.replace(r'\s*·\s*', '', regex=True)
                   .str.replace('–', '-', regex=False))
    df_clean[c] = df_clean[c].where(df_clean[c].notna() & (df_clean[c] != ''), np.nan)

# Canonicalise "nothing / don't play" tokens in multi-select columns
nothing_pattern = re.compile(
    r'^\s*(ни\s*во\s*что.*|ничего.*|не\s+играю.*|нет\s+играю.*|я\s+не\s+играю.*)$',
    re.IGNORECASE,
)

def normalise_multiselect(s):
    if pd.isna(s) or s == '':
        return ''
    parts = [p.strip() for p in str(s).split(';') if p.strip()]
    out = []
    for p in parts:
        if nothing_pattern.match(p):
            out.append('ничего из перечисленного')
        else:
            out.append(p)
    seen, deduped = set(), []
    for p in out:
        if p not in seen:
            seen.add(p); deduped.append(p)
    return '; '.join(deduped)

for c in multi_select_vars:
    df_clean[c] = df_clean[c].map(normalise_multiselect)

print("Sample about_you values:")
print(df_clean['about_you'].replace('', np.nan).dropna().value_counts().head(8).to_string())

In [ ]:
df_clean['flag_short_time'] = df_clean['time_spent'] < 60
df_clean['flag_long_time']  = df_clean['time_spent'] > 3600
df_clean['flag_test_row']   = (
    df_clean['expectations'].astype('string').str.contains('тест', case=False, na=False)
    | df_clean['conditions_desc'].astype('string').str.contains('ыкцкручеру', case=False, na=False)
)

print("Flag counts:")
print(df_clean[['flag_short_time', 'flag_long_time', 'flag_test_row']].sum().to_string())

print("\nRows flagged as test data:")
print(df_clean.loc[df_clean['flag_test_row'],
                   ['response_id', 'course', 'program', 'expectations']].to_string(index=False))

print("\nRows with time_spent > 1 hour:")
print(df_clean.loc[df_clean['flag_long_time'],
                   ['response_id', 'time_spent', 'course', 'program']].to_string(index=False))

In [ ]:
OUT_CSV = 'autumn_clean.csv'
df_clean.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f"Wrote {OUT_CSV}  shape={df_clean.shape}")

In [ ]:
question_text_for = {}
for ci in columns_info:
    qnum = ci['qnum']
    if qnum is None:
        continue
    block = get_block(qnum)
    if block is None:
        continue
    rel = qnum if block == 1 else qnum - 23 * (block - 1)
    var_map = block1_map if block == 1 else block2_map
    if rel not in var_map:
        continue
    name = var_map[rel]
    if name not in question_text_for:
        question_text_for[name] = ci['question_text']

meta_desc = {
    'response_id': 'Response ID (integer)',
    'date':        'Date of response (datetime)',
    'time_spent':  'Time spent on survey, seconds (integer)',
    'ip':          'IP address (blank in source)',
    'source':      'Distribution source',
    'os':          'Operating system (blank in source)',
    'browser':     'Browser (blank in source)',
    'device':      'Device (blank in source)',
}
flag_desc = {
    'flag_short_time': 'True if time_spent < 60 s',
    'flag_long_time':  'True if time_spent > 3600 s',
    'flag_test_row':   'True if row looks like test / junk data',
}

lines = [
    "# Codebook for Cleaned Survey Data",
    "",
    f"**Rows:** {len(df_clean)}  ",
    f"**Columns:** {len(df_clean.columns)}  ",
    "",
    "Source: `autumn.csv` — five concatenated survey blocks unified into one row per respondent.",
    "",
    "**Conventions:**",
    "- Multi-select variables: semicolon-separated (`; `), empty string `''` if nothing selected.",
    "- Single-choice variables: value or `NaN` (empty cell in CSV) if not answered.",
    "",
    "## Metadata variables",
    "",
    "| Variable | Description | Type |",
    "|----------|-------------|------|",
]
for c in metadata_order:
    lines.append(f"| `{c}` | {meta_desc.get(c, c)} | metadata |")
lines += ["", "## Quality flags", "",
          "| Variable | Description |", "|----------|-------------|"]
for c, d in flag_desc.items():
    lines.append(f"| `{c}` | {d} |")
lines += ["", "## Survey variables", ""]

def describe_values(var, top_n=12):
    s = df_clean[var]
    if var not in multi_select_vars:
        s = s.dropna()
    else:
        s = s.replace('', np.nan).dropna()
    n = len(s)
    if n == 0:
        return "_no non-empty values_"
    if var in multi_select_vars:
        exploded = s.astype(str).str.split('; ').explode()
        vc = exploded.value_counts().head(top_n)
        out = [f"_multi-select, {n} responses, {len(exploded)} selections_", "",
               "| Option | Count |", "|--------|-------|"]
        for k, v in vc.items():
            out.append(f"| {k} | {v} |")
        return "\n".join(out)
    vc = s.value_counts().head(top_n)
    out = [f"_n = {n}_", "", "| Value | Count |", "|-------|-------|"]
    for k, v in vc.items():
        k_clean = str(k).replace('|', '\\|')
        out.append(f"| {k_clean} | {v} |")
    return "\n".join(out)

for var in variable_order:
    qtext = question_text_for.get(var, '')
    is_multi = var in multi_select_vars
    lines.append(f"### `{var}`")
    lines.append("")
    if qtext:
        lines += [f"**Question:** {qtext}", ""]
    lines += [f"**Type:** {'multi-select' if is_multi else 'single / text'}", ""]
    lines.append(describe_values(var))
    lines.append("")

with open('codebook.md', 'w', encoding='utf-8') as f:
    f.write("\n".join(lines))
print("Wrote codebook.md")

In [ ]:
assert len(df_clean) == len(df_raw), "Row count mismatch!"
print(f"✅ Row count OK: {len(df_clean)}")

assert df_clean.columns[:len(metadata_order)].tolist() == metadata_order
print("✅ Metadata column order OK")

# Multi-select integrity
for c in multi_select_vars:
    s = df_clean[c].astype(str)
    bad = s.str.startswith(';') | s.str.endswith(';')
    assert not bad.any(), f"Malformed multi-select in {c}"
print("✅ Multi-select columns well-formed")

# No literal 'nan' strings snuck in
for c in text_cols:
    n_nanstr = (df_clean[c].astype(str) == 'nan').sum()
    if n_nanstr:
        print(f"⚠️  {c} contains {n_nanstr} literal 'nan' strings")
print("✅ No literal 'nan' strings in text columns")

# Spot-checks
print("\nRespondent 1:")
print(df_clean.loc[df_clean['response_id'] == 1,
                   ['course','program','difficulties','priorities']].T.to_string(header=False))
print("\nRespondent 2 games_team (Minecraft should appear from write-in):")
print(df_clean.loc[df_clean['response_id'] == 2, 'games_team'].iloc[0])